# Section 1: Imports and Setup
This section sets up the environment, loads API keys from .env, and prepares the DeepLenseSim + agent workflow modules.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'models.py').exists():
    PROJECT_DIR = Path.cwd() / 'Specific_Test_II'

sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

load_dotenv(PROJECT_DIR / '.env')

print(f'Working directory: {PROJECT_DIR}')
print(f'OPENAI_API_KEY loaded: {bool(os.getenv("OPENAI_API_KEY"))}')

# Section 2: Pydantic Models Overview
This section imports the Pydantic models and displays their schemas to verify the expected structured inputs and outputs.

In [ ]:
from models import AgentState, ClarificationRequest, SimulationOutput, SimulationParams

print('SimulationParams schema:')
print(json.dumps(SimulationParams.model_json_schema(), indent=2))

print('\nSimulationOutput schema:')
print(json.dumps(SimulationOutput.model_json_schema(), indent=2))

# Section 3: Tool Functions Demo
This section demonstrates prompt parsing and validation before running the state graph.

In [ ]:
from tools import parse_user_prompt, simulation_output_json, validate_params

demo_prompt = 'Generate 5 lensing images with subhalo substructure, source redshift 1.5, lens redshift 0.5, Model_I'
parsed = parse_user_prompt(demo_prompt)
validated = validate_params(parsed)

print('Parsed:')
print(parsed)
print('\nValidated object type:', type(validated).__name__)
print(validated)

# Section 4: LangGraph State Machine
This section builds the LangGraph workflow with human-in-the-loop clarification and defines a reusable runner helper.

In [ ]:
from datetime import datetime

from langgraph.types import Command

from graph import build_graph

graph = build_graph()

def run_agent(prompt: str, auto_clarification: str | None = None):
    # Run the graph and handle human-in-the-loop interruptions until completion.
    config = {'configurable': {'thread_id': f'deeplense-{datetime.now().timestamp()}'}}

    # Initial invoke with user prompt.
    result = graph.invoke({'user_prompt': prompt, 'messages': []}, config=config)

    # Keep resuming if the graph asks for clarification.
    while '__interrupt__' in result:
        interrupt_payload = result['__interrupt__'][0].value
        print('Clarification requested:')
        for q in interrupt_payload.get('questions', []):
            print('-', q)

        if auto_clarification is not None:
            reply = auto_clarification
            print(f'Auto clarification used: {reply}')
            auto_clarification = None
        else:
            reply = input('Your clarification: ')

        # Resume graph execution with the user's clarification text.
        result = graph.invoke(Command(resume=reply), config=config)

    return result

# Section 5: Demo 1 - Complete Prompt
This section runs the full agent flow from a complete prompt and prints the final SimulationOutput in formatted JSON.

In [ ]:
prompt_complete = (
    'Generate 5 lensing images with subhalo substructure, '
    'source redshift 1.5, lens redshift 0.5, Model_I'
)

demo1_result = run_agent(prompt_complete)
demo1_output = demo1_result.get('simulation_output')

if demo1_output is not None and hasattr(demo1_output, 'model_dump'):
    print(json.dumps(demo1_output.model_dump(), indent=2))
else:
    print(json.dumps(demo1_output, indent=2))

# Section 6: Demo 2 - Incomplete Prompt (triggers HITL)
This section sends an incomplete request so the graph pauses for clarification and waits for user input before continuing.

In [ ]:
prompt_incomplete = 'Generate some lensing images'

# Leave auto_clarification=None to force true human-in-the-loop behavior via input().
demo2_result = run_agent(prompt_incomplete, auto_clarification=None)
demo2_output = demo2_result.get('simulation_output')

if demo2_output is not None and hasattr(demo2_output, 'model_dump'):
    print(json.dumps(demo2_output.model_dump(), indent=2))
else:
    print(json.dumps(demo2_output, indent=2))

# Section 7: Display Generated Images inline
This section displays generated PNG images inline from the output metadata and the outputs folder.

In [ ]:
from math import ceil

def display_generated_images(result_obj, max_images=12):
    # Display generated image files from SimulationOutput metadata.
    output = result_obj.get('simulation_output') if isinstance(result_obj, dict) else None
    if output is None:
        print('No simulation output available.')
        return

    if hasattr(output, 'model_dump'):
        output_dict = output.model_dump()
    else:
        output_dict = output

    image_paths = output_dict.get('image_paths', [])[:max_images]
    if not image_paths:
        print('No image paths found in output.')
        return

    cols = 4
    rows = ceil(len(image_paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))

    if rows == 1:
        axes = [axes] if cols == 1 else axes

    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, ax in enumerate(axes_flat):
        if i < len(image_paths):
            img = Image.open(image_paths[i]).convert('L')
            ax.imshow(img, cmap='gray')
            ax.set_title(Path(image_paths[i]).name)
            ax.axis('off')
        else:
            ax.axis('off')

    plt.tight_layout()
    plt.show()

# Display images from the complete-prompt run first.
display_generated_images(demo1_result)